# Validation
---

## Package(s)

In [15]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Helper Function(s)

In [16]:
def make_partitions(pre_dim: int, pooled_dim: int):
    """Partition [pre_dim] into pooled_dim non-empty contiguous blocks."""
    assert pre_dim >= pooled_dim
    base = pre_dim // pooled_dim
    rem = pre_dim % pooled_dim

    parts = []
    start = 0
    for k in range(pooled_dim):
        size = base + (1 if k < rem else 0)
        parts.append(torch.arange(start, start + size, dtype=torch.long))
        start += size
    return parts


class NonnegativeLinear(nn.Module):
    """Linear layer with entrywise nonnegative weights via softplus."""
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.raw_weight = nn.Parameter(
            torch.empty(out_features, in_features).normal_(mean=-2.0, std=0.25)
        )
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

    @property
    def weight(self):
        return F.softplus(self.raw_weight)

    def forward(self, x):
        return F.linear(x, self.weight, self.bias)


class GatedPReLU(nn.Module):
    r"""
    PReLU_alpha(u) = ReLU(u) - alpha ReLU(-u),
    where alpha in [0,1].
    """
    def __init__(self, dim: int, init_alpha: float = 0.0):
        super().__init__()
        init_alpha = min(max(init_alpha, 1e-4), 1.0 - 1e-4)
        raw = math.log(init_alpha / (1.0 - init_alpha))
        self.raw_alpha = nn.Parameter(torch.full((dim,), raw))

    @property
    def alpha(self):
        return torch.sigmoid(self.raw_alpha)

    def forward(self, x):
        return F.relu(x) - self.alpha * F.relu(-x)


class MaxPoolPartition(nn.Module):
    """Blockwise max-pooling according to a fixed partition."""
    def __init__(self, pre_dim: int, pooled_dim: int):
        super().__init__()
        parts = make_partitions(pre_dim, pooled_dim)
        self.num_parts = len(parts)
        for k, idx in enumerate(parts):
            self.register_buffer(f"idx_{k}", idx)

    def forward(self, z):
        pooled = []
        for k in range(self.num_parts):
            idx = getattr(self, f"idx_{k}")
            pooled.append(z.index_select(dim=1, index=idx).max(dim=1).values)
        return torch.stack(pooled, dim=1)


## Models

In [17]:
class CNFLayer(nn.Module):
    r"""
    x^{ell+1}
      = max_{Pi^{ell}} PReLU_alpha^{bullet}(A^{ell} x^{ell} + b^{ell})
    with A^{ell} >= 0 and alpha in [0,1].
    """
    def __init__(self, in_dim: int, pooled_dim: int, expansion: int = 2, init_alpha: float = 0.0):
        super().__init__()
        pre_dim = expansion * pooled_dim
        assert pre_dim >= pooled_dim

        self.linear = NonnegativeLinear(in_dim, pre_dim, bias=True)
        self.act = GatedPReLU(pre_dim, init_alpha=init_alpha)
        self.pool = MaxPoolPartition(pre_dim, pooled_dim)

    def forward(self, x):
        z = self.linear(x)
        z = self.act(z)
        z = self.pool(z)
        return z


class CNF(nn.Module):
    r"""
    Finite-dimensional implementation of the CNF definition over H = R^input_dim.

    Input layer:
        x^(0)_m = <p_m, x> + constant_shift.

    Hidden layers:
        nonnegative affine -> gated PReLU -> max-pooling.

    Final layer:
        nonnegative affine to R.
    """
    def __init__(
        self,
        input_dim: int,
        M: int,
        pooled_dims=(64, 32),
        expansion: int = 2,
        init_alpha: float = 0.0,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.M = M

        # Trainable measurement vectors p_1,...,p_M in R^input_dim.
        self.p = nn.Parameter(torch.randn(M, input_dim) / math.sqrt(input_dim))

        # This scalar represents the Psi_N(...) constant in the definition.
        self.constant_shift = nn.Parameter(torch.tensor(0.0))

        dims = [M] + list(pooled_dims)
        self.layers = nn.ModuleList([
            CNFLayer(dims[i], dims[i + 1], expansion=expansion, init_alpha=init_alpha)
            for i in range(len(dims) - 1)
        ])

        self.final = NonnegativeLinear(dims[-1], 1, bias=True)

    def forward(self, x):
        x0 = x @ self.p.T + self.constant_shift
        z = x0
        for layer in self.layers:
            z = layer(z)
        return self.final(z).squeeze(-1)


# ------------------------------------------------------------
# Random convex target: larger positive-weight ReLU MLP
# ------------------------------------------------------------

class RandomConvexReLUTarget(nn.Module):
    r"""
    A random convex ReLU MLP.

    First layer has arbitrary affine responses:
        ReLU(W_0 x + b_0), convex coordinatewise because ReLU o affine is convex.

    Subsequent layers use nonnegative weights:
        ReLU(W_l z + b_l), W_l >= 0.

    Final layer is a nonnegative linear combination.
    """
    def __init__(self, input_dim: int, hidden=(128, 128), seed: int = 123):
        super().__init__()
        torch.manual_seed(seed)

        dims = [input_dim] + list(hidden)

        self.W = nn.ParameterList()
        self.b = nn.ParameterList()

        # First layer: arbitrary weights.
        W0 = torch.randn(dims[1], dims[0]) / math.sqrt(dims[0])
        b0 = torch.randn(dims[1]) * 0.2
        self.W.append(nn.Parameter(W0, requires_grad=False))
        self.b.append(nn.Parameter(b0, requires_grad=False))

        # Hidden layers: nonnegative weights.
        for i in range(1, len(dims) - 1):
            Wi = torch.rand(dims[i + 1], dims[i]) / math.sqrt(dims[i])
            bi = torch.randn(dims[i + 1]) * 0.2
            self.W.append(nn.Parameter(Wi, requires_grad=False))
            self.b.append(nn.Parameter(bi, requires_grad=False))

        # Final layer: nonnegative row vector.
        Wout = torch.rand(1, dims[-1]) / math.sqrt(dims[-1])
        bout = torch.randn(1) * 0.2
        self.Wout = nn.Parameter(Wout, requires_grad=False)
        self.bout = nn.Parameter(bout, requires_grad=False)

    def forward(self, x):
        z = F.relu(F.linear(x, self.W[0], self.b[0]))
        for W, b in zip(self.W[1:], self.b[1:]):
            z = F.relu(F.linear(z, W, b))
        return F.linear(z, self.Wout, self.bout).squeeze(-1)


# ------------------------------------------------------------
# Convexity sanity check
# ------------------------------------------------------------

@torch.no_grad()
def max_jensen_violation(model, input_dim: int, n_tests: int = 4096, radius: float = 2.0):
    x = radius * (2 * torch.rand(n_tests, input_dim) - 1)
    y = radius * (2 * torch.rand(n_tests, input_dim) - 1)
    t = torch.rand(n_tests, 1)

    lhs = model(t * x + (1 - t) * y)
    rhs = t.squeeze(-1) * model(x) + (1 - t.squeeze(-1)) * model(y)

    return (lhs - rhs).max().item()




## Run Experiment

In [ ]:
d_in = 1
MLP_width = int(500)
ratio_param = .2
CFN_width = 2 * int(round(MLP_width * ratio_param))

# ------------------------------------------------------------
# Toy experiment
# ------------------------------------------------------------

def count_params(model):
    return sum(p.numel() for p in model.parameters())


def main():
    torch.manual_seed(0)

    input_dim = d_in
    n_train = 2048
    n_test = 1024

    target = RandomConvexReLUTarget(input_dim=input_dim, hidden=(MLP_width, MLP_width), seed=2026)
    model = CNF(
        input_dim=input_dim,
        M=CFN_width // 2,
        pooled_dims=(CFN_width, CFN_width // 2),
        expansion=2,
        init_alpha=0.0,
    )

    target_params = count_params(target)
    model_params = count_params(model)

    print(f"Random convex ReLU MLP target parameters: {target_params:,}")
    print(f"CNF trainable parameters:                 {model_params:,}")
    print(f"CNF / target parameter ratio:             {model_params / target_params:.3f}")

    x_train = 4 * torch.rand(n_train, input_dim) - 2
    y_train = target(x_train).detach()

    x_test = torch.linspace(-2, 2, n_test).reshape(-1, 1)
    y_test = target(x_test).detach()

    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)

    for step in range(3001):
        pred = model(x_train)
        loss = F.mse_loss(pred, y_train)

        opt.zero_grad()
        loss.backward()
        opt.step()

        if step % 500 == 0:
            with torch.no_grad():
                test_loss = F.mse_loss(model(x_test), y_test).item()
                viol = max_jensen_violation(model, input_dim)
            print(
                f"step={step:4d} | train_mse={loss.item():.4e} "
                f"| test_mse={test_loss:.4e} | max_jensen_violation={viol:.4e}"
            )

    with torch.no_grad():
        y_hat = model(x_test)

    plt.figure(figsize=(7, 4))
    plt.plot(x_test.squeeze().numpy(), y_test.numpy(), label="random convex ReLU target")
    plt.plot(x_test.squeeze().numpy(), y_hat.numpy(), "--", label="trained CNF")
    plt.scatter(x_train[:150].squeeze().numpy(), y_train[:150].numpy(), s=8, alpha=0.25, label="samples")
    plt.legend()
    plt.title(
        f"CNF fit to a random convex ReLU MLP target\n"
        f"target params: {target_params:,} | CNF params: {model_params:,}"
    )
    plt.xlabel("x")
    plt.ylabel("value")
    plt.tight_layout()
    plt.show()

    print("Target Jensen violation:", max_jensen_violation(target, input_dim))
    print("CNF Jensen violation:   ", max_jensen_violation(model, input_dim))


if __name__ == "__main__":
    main()

Random convex ReLU MLP target parameters: 252,001
CNF trainable parameters:                 81,402
CNF / target parameter ratio:             0.323
step=   0 | train_mse=5.6329e+03 | test_mse=4.7095e+03 | max_jensen_violation=7.6294e-05


---

# Fin

---